In [3]:
import os
import json
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("API"))

In [4]:
def load_user_messages(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    username = data.get("username")
    messages = data.get("messages", [])
    
    return username, messages

In [22]:
def build_analysis_prompt(username, messages):

    message_block = "\n".join([f"{i+1}. {msg}" for i, msg in enumerate(messages)])

    prompt = f"""
You are analyzing Reddit posts/comments from ONE user account.

Username: {username}

Messages written by this account:

{message_block}

Analyze the messages collectively to detect patterns, intent, and likelihood of automation.

Tasks:

1. Identify whether the messages promote a repeated theme, narrative, ideology, or running joke.

2. Classify the primary content intent:

- spam (advertising, scams, promotion, link farming)
- propaganda (political or ideological narrative pushing)
- normal_discussion (general conversation across varied topics)
- running_bit (repeated joke, meme persona, or intentionally recurring theme)

3. Determine whether the account appears to be:
- human_user
- likely_bot

Important:
Spam or propaganda can be posted by both humans and bots.  
When estimating automation likelihood, focus on signals such as:
- repetitive or templated messages
- identical structure across posts
- lack of conversational engagement
- copy-paste style content
- systematic or mechanical tone.

4. Estimate the probability that the account is automated.

5. If likely_bot, classify the most probable bot type:

- spam_bot
- propaganda_bot
- karma_farming_bot
- engagement_bot
- unknown_bot

Return ONLY JSON in the following format:

{{
"content_classification": "spam | propaganda | normal_discussion | running_bit",
"account_type": "human_user | likely_bot",
"pattern_detected": true/false,
"pattern_description": "...",
"bot_probability": 0.0,
"bot_type": "spam_bot | propaganda_bot | karma_farming_bot | engagement_bot | unknown_bot",
"key_indicators": [
"indicator1",
"indicator2",
"indicator3"
]
}}
"""
    return prompt

In [6]:
def analyze_user(messages, username):

    prompt = build_analysis_prompt(username, messages)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [23]:
username, messages = load_user_messages("reddit_user_scraped.json")

In [24]:
result = analyze_user(messages, username)

print(result)

```json
{
  "content_classification": "normal_discussion",
  "account_type": "human_user",
  "pattern_detected": true,
  "pattern_description": "The user frequently engages in detailed discussions across two primary domains: competitive gaming (specifically Genshin Impact character builds, team compositions, gacha mechanics, and Clash of Clans attack strategies) and academic/career development (seeking advice on resume improvement, AI/ML masters programs from 'tier-3' colleges, and identifying job scams). There's a consistent pattern of asking for advice, giving specific recommendations, and sharing personal experiences within these niche areas.",
  "bot_probability": 0.01,
  "bot_type": "unknown_bot",
  "key_indicators": [
    "Highly specific and technical language across diverse topics (e.g., 'autoencoder and cnn', 'Neuvillete C0R1 rotation', 'zap witches th 10 strategy').",
    "Exhibits personal experiences and opinions ('I was thinking of applying for tum', 'I wanted xingque and 

In [26]:
username, messages = load_user_messages("reddit_user_scraped_bot.json")

In [27]:
result = analyze_user(messages, username)

print(result)

```json
{
  "content_classification": "running_bit",
  "account_type": "likely_bot",
  "pattern_detected": true,
  "pattern_description": "All comments (excluding the initial 'About Me' post) follow an identical template. They quote another user's message, then state 'I detect haikus. And sometimes, successfully.Learn more about me.Opt out of replies: \"haikusbot opt out\" | Delete my comment: \"haikusbot delete\"'. The core pattern is detecting haikus in other users' comments and providing a templated response, acting as a recurring 'bit' or functional response.",
  "bot_probability": 1.0,
  "bot_type": "engagement_bot",
  "key_indicators": [
    "Highly repetitive and templated message structure across all comments.",
    "Username 'haikusbot' explicitly contains 'bot'.",
    "Phrasing like 'I detect haikus' indicates an automated process.",
    "Inclusion of bot-specific instructions like 'Opt out of replies: \"haikusbot opt out\" | Delete my comment: \"haikusbot delete\"'.",
    "L